### Experiment 5: Encrypt the plaintext message using RSA algorithm. Then perform the reverse operation to get original plaintext.

In [1]:
# ============================================================
# RSA ALGORITHM (Rivest–Shamir–Adleman)
# ------------------------------------------------------------
# RSA is an ASYMMETRIC encryption algorithm — it uses TWO keys:
#   - PUBLIC KEY  (e, n): shared openly, used to ENCRYPT
#   - PRIVATE KEY (d, n): kept secret,   used to DECRYPT
#
# SECURITY BASIS:
#   It is easy to multiply two large primes p and q to get n,
#   but extremely hard to REVERSE (factor n back into p and q).
#   This 'factoring problem' is what makes RSA secure.
#
# RSA KEY GENERATION STEPS:
#   1. Choose two large prime numbers p and q
#   2. Compute n = p * q          (the modulus, public)
#   3. Compute phi = (p-1)*(q-1)  (Euler's totient, kept secret)
#   4. Choose e such that: 1 < e < phi AND gcd(e, phi) = 1
#      (e is the public exponent)
#   5. Compute d = modular inverse of e mod phi
#      (d is the private exponent; e*d ≡ 1 mod phi)
#
# ENCRYPTION:  C = M^e mod n   (M = message, C = ciphertext)
# DECRYPTION:  M = C^d mod n   (recover original message)
# ============================================================

import random
from math import gcd, isqrt


# ----------------------------------------------------------
# HELPER: Primality Test
# Checks if n is prime by trial division up to sqrt(n)
# ----------------------------------------------------------
def is_prime(n):
    if n < 2:
        return False   # 0 and 1 are not prime
    # Only check divisors up to sqrt(n) — any factor larger
    # than sqrt(n) would require a paired factor smaller than it
    for i in range(2, isqrt(n) + 1):
        if n % i == 0:
            return False   # found a divisor -> not prime
    return True   # no divisors found -> prime


# ----------------------------------------------------------
# KEY GENERATION
# Produces the RSA public and private key pair
# ----------------------------------------------------------
def generate_keypair():

    # STEP 1: Build a list of all primes between 50 and 200
    primes = [p for p in range(50, 200) if is_prime(p)]

    # STEP 2: Randomly pick two DISTINCT primes p and q
    p = random.choice(primes)
    q = random.choice(primes)
    while p == q:          # make sure p and q are different
        q = random.choice(primes)

    # STEP 3: Compute n = p * q
    # n is the MODULUS — part of both public and private keys
    # Its size (bit length) determines RSA's security strength
    n = p * q

    # STEP 4: Compute Euler's Totient phi(n) = (p-1) * (q-1)
    # phi counts how many integers in [1, n] are coprime to n
    # This value is kept SECRET (knowing it allows breaking RSA)
    phi = (p - 1) * (q - 1)

    # STEP 5: Choose the public exponent e
    # e = 65537 is a standard choice in real RSA (it's prime and
    # has a small number of 1-bits, making encryption fast)
    # Requirement: gcd(e, phi) == 1  (e and phi must be coprime)
    e = 65537
    while gcd(e, phi) != 1:   # if 65537 doesn't work, pick a random valid e
        e = random.randint(3, phi - 1)

    # STEP 6: Compute d = modular inverse of e mod phi
    # This satisfies:  e * d ≡ 1 (mod phi)
    # pow(e, -1, phi) is Python 3.8+'s built-in modular inverse
    d = pow(e, -1, phi)

    # Return: public key (e, n), private key (d, n), and the primes
    return (e, n), (d, n), p, q


# ----------------------------------------------------------
# ENCRYPTION: C = M^e mod n
# Takes a numeric message M and the public key
# Returns the ciphertext C
# IMPORTANT: M must be less than n to work correctly
# ----------------------------------------------------------
def rsa_encrypt(message, public_key):
    e, n = public_key                 # unpack public key
    return pow(message, e, n)         # fast modular exponentiation: M^e mod n


# ----------------------------------------------------------
# DECRYPTION: M = C^d mod n
# Takes the ciphertext C and the private key
# Returns the original message M
# ----------------------------------------------------------
def rsa_decrypt(ciphertext, private_key):
    d, n = private_key                # unpack private key
    return pow(ciphertext, d, n)      # fast modular exponentiation: C^d mod n


# ----------------------------------------------------------
# HELPER: Convert string to list of ASCII values
# RSA works on NUMBERS, so text must be converted first
# e.g., 'A' -> 65,  'H' -> 72,  'E' -> 69
# ----------------------------------------------------------
def string_to_ascii(text):
    return [ord(c) for c in text]   # ord() gives the ASCII code of each character


# ----------------------------------------------------------
# HELPER: Convert list of ASCII values back to string
# e.g., [72, 69, 76, 76, 79] -> 'HELLO'
# ----------------------------------------------------------
def ascii_to_string(ascii_list):
    return ''.join(chr(i) for i in ascii_list)   # chr() converts ASCII code to character


# ----------------------------------------------------------
# MAIN: Generate keys and display them
# ----------------------------------------------------------
public_key, private_key, p, q = generate_keypair()
e, n = public_key    # extract public exponent and modulus
d, _ = private_key   # extract private exponent (n is same for both)

print("RSA Key Generation:")
print(f"Prime p: {p}")                  # one of the secret primes
print(f"Prime q: {q}")                  # the other secret prime
print(f"n = {n}")                       # modulus = p * q (public)
print(f"Public key: ({e}, {n})")        # (public exponent, modulus)
print(f"Private key: ({d}, {n})")       # (private exponent, modulus)

RSA Key Generation:
Prime p: 109
Prime q: 79
n = 8611
Public key: (65537, 8611)
Private key: (881, 8611)


In [2]:
# ============================================================
# APPLYING RSA: Integer and String Encryption/Decryption
# ============================================================

# ----------------------------------------------------------
# PART 1: Integer Encryption
# Directly encrypt a small integer (must be < n)
# ----------------------------------------------------------
integer_message = 42    # the plaintext message (just a number)

# Encrypt: compute 42^e mod n  -> produces a ciphertext number
encrypted_int = rsa_encrypt(integer_message, public_key)

# Decrypt: compute ciphertext^d mod n -> should recover 42
decrypted_int = rsa_decrypt(encrypted_int, private_key)

print("INTEGER ENCRYPTION:")
print(f"Original: {integer_message}, Encrypted: {encrypted_int}, Decrypted: {decrypted_int}")
print(f"Success: {integer_message == decrypted_int}\n")   # True if decryption is correct


# ----------------------------------------------------------
# PART 2: String Encryption
# Strings can't be encrypted directly — convert to ASCII first
# Each character is encrypted INDEPENDENTLY as its ASCII value
# ----------------------------------------------------------
string_message = "HELLO"   # the plaintext string

# STEP 1: Convert each character to its ASCII integer
# 'H'->72, 'E'->69, 'L'->76, 'L'->76, 'O'->79
ascii_vals = string_to_ascii(string_message)

# STEP 2: Encrypt each ASCII value separately using the public key
# Each number M becomes: M^e mod n
encrypted = [rsa_encrypt(val, public_key) for val in ascii_vals]

# STEP 3: Decrypt each cipher value using the private key
# Each ciphertext C becomes: C^d mod n -> recovers original ASCII
decrypted = [rsa_decrypt(val, private_key) for val in encrypted]

# STEP 4: Convert the decrypted ASCII values back to a string
decrypted_string = ascii_to_string(decrypted)

print("STRING ENCRYPTION:")
print(f"Original: '{string_message}'")        # original text
print(f"Encrypted: {encrypted}")              # list of ciphertext numbers
print(f"Decrypted: '{decrypted_string}'")     # recovered text
print(f"Success: {string_message == decrypted_string}")  # True if fully recovered

INTEGER ENCRYPTION:
Original: 42, Encrypted: 287, Decrypted: 42
Success: True

STRING ENCRYPTION:
Original: 'HELLO'
Encrypted: [8297, 4615, 4218, 4218, 3871]
Decrypted: 'HELLO'
Success: True
